In [0]:
from pyspark.sql import SparkSession

In [0]:
spark=(
    SparkSession
    .builder
    .appName("Getting Hands on Distinct, Aggs & Window Functions")
    .getOrCreate()
)


In [0]:
emp_data = [
    # Engineering (201) - Head: E01
    ["E01","201","Kabir Singh","45","Male","95000","2011-01-15",None],
    ["E02","201","Meera Rao","34","Female","80000","2015-03-10","E01"],
    ["E03","201","Rohan Das","33","Male","80000","2016-07-22","E01"],   # salary tie with E02
    ["E04","201","Priya Nair","29","Female","62000","2018-02-01","E01"],
    ["E05","201","Arjun Mehta","27","Male","58000","2019-06-15","E01"],
    ["E06","201","Divya Kapoor","28","Female","58000","2020-01-10","E01"], # salary tie with E05

    # Sales (202) - Head: E07
    ["E07","202","Sanjay Gupta","42","Male","90000","2012-04-01",None],
    ["E08","202","Neha Verma","31","Female","70000","2016-09-01","E07"],
    ["E09","202","Vikram Joshi","36","Male","65000","2014-11-11","E07"],
    ["E10","202","Anjali Sharma","30","Female","65000","2017-05-05","E07"], # salary tie with E09
    ["E11","202","Rahul Kumar","26","Male","50000","2021-08-20","E07"],

    # HR (203) - Head: E12
    ["E12","203","Sunita Iyer","40","Female","85000","2013-02-14",None],
    ["E13","203","Kavita Menon","35","Female","60000","2015-12-01","E12"],
    ["E14","203","Deepak Chawla","33","Male","55000","2018-10-10","E12"],
    ["E15","203","Pooja Bhatt","29","Female","55000","2019-03-03","E12"],  # salary tie with E14

    # Finance (204) - Head: E16
    ["E16","204","Ramesh Pillai","48","Male","98000","2010-01-01",None],
    ["E17","204","Anita Desai","37","Female","72000","2014-06-01","E16"],
    ["E18","204","Suresh Reddy","39","Male","68000","2013-09-09","E16"],
    ["E19","204","Lakshmi Menon","34","Female","68000","2016-04-04","E16"], # salary tie with E18
    ["E20","204","Manoj Tiwari","26","Male","51000","2020-11-11","E16"],

    # Marketing (205) - Head: E21
    ["E21","205","Farah Khan","41","Female","88000","2012-07-07",None],
    ["E22","205","Imran Ali","30","Male","61000","2017-01-01","E21"],
    ["E23","205","Zoya Sheikh","28","Female","61000","2019-09-09","E21"],  # salary tie with E22
    ["E24","205","Tariq Ahmed","25","Male","45000","2022-02-02","E21"],

    # Single-employee department (206) - no manager, tests edge cases
    ["E25","206","Contractor X","50","Male","40000","2021-01-01",None],

    # Duplicate rows (EXACT copies) -> for dedup practice
    ["E04","201","Priya Nair","29","Female","62000","2018-02-01","E01"],
    ["E13","203","Kavita Menon","35","Female","60000","2015-12-01","E12"],
]

emp_schema = """
    employee_id string,
    department_id string,
    name string,
    age string,
    gender string,
    salary string,
    hire_date string,
    manager_id string
"""

emp = spark.createDataFrame(emp_data, schema=emp_schema)
emp.show(30, truncate=False)

In [0]:
emp.count()

In [0]:
emp.show()

In [0]:
from pyspark.sql.functions import col


emp_clean= emp.select(col("employee_id")).distinct()

In [0]:
emp_clean.count()

In [0]:
## Want to pick employee based on top salaries in each department


from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, row_number, rank, expr


WindowSpec= Window.partitionBy("department_id").orderBy(col("salary").desc())
RowNumberFunc= row_number().over(WindowSpec)
RankFunc= rank().over(WindowSpec)
DenseRankFunc= dense_rank().over(WindowSpec)

emp_sal_classification=emp.withColumn("SalaryOrder", RowNumberFunc).withColumn("SalaryRank", RankFunc).withColumn("SalaryDenseRank", DenseRankFunc)
emp_sal_classification.where("SalaryDenseRank=1").show(30, truncate=False)




In [0]:

## Employees with lowest salaries in each department

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, row_number, rank, expr

WindowSpec = Window.partitionBy("department_id").orderBy(col("salary").asc())

RowNumberFunc= row_number().over(WindowSpec)
RankFunc= rank().over(WindowSpec)
DenseRankFunc= dense_rank().over(WindowSpec)

emp_with_lowest_sal= emp.withColumn("SalaryRank", DenseRankFunc)
emp_with_lowest_sal.where("SalaryRank=1").show()


In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col

WindowSpec= Window.partitionBy("department_id")
AvgSalary= avg(col("salary")).over(WindowSpec)

emp_avg_salary= emp.withColumn("AvgSalary", AvgSalary)
emp_avg_salary.select("department_id","AvgSalary").distinct().show()

In [0]:
emp.show()

In [0]:
emp.show()

In [0]:

emp.where(col("manager_id").isNull()).show()

In [0]:
empManagementLevels= emp.withColumn("ManagementLevel", 
                                    expr("case when manager_id is null then 'Delivery Manager' when department_id=201 then 'Data Consultant' when department_id=202 then 'Site Reliablility Engineer' when department_id=203 then 'Data Scientist' when department_id=204 then 'Cloud Ops' else 'Operations Consultant' end"))
empManagementLevels.show()

In [0]:
## Top 3 salaries in each department

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col


WindowSpec= Window.partitionBy("department_id").orderBy(col("salary").desc())
DenseRankFunc= dense_rank().over(WindowSpec)

emp_top3sal= emp.withColumn("SalaryDenseRank", DenseRankFunc)
emp_top3sal.where("SalaryDenseRank<=3").show()

